# 20261102 串讲 · LangChain 工具调用（Tool Calling）

6 个 `test*.py` 沿着一条主线展开：

> 模型如何提出工具请求，程序如何执行，再把结果交回模型？

一次完整任务通常经历：**描述工具 → 模型提出调用 → 程序校验并执行 → 回填结果 → 模型继续决策或回答**。
一次模型请求只负责生成本次响应；执行器负责推动整个循环。

| 步 | 对应文件 | 工具协议由谁描述 | 调用内容由谁生成 | 谁执行并推进循环 |
|---|---|---|---|---|
| 1 | `test1.py` | prompt + Pydantic 格式说明 | 模型在正文中生成 JSON | 自己写 `chat()` |
| 2 | `test2.py` | `bind_tools()` 绑定工具 schema | 模型返回原生工具调用 | 原脚本只演示一次请求 |
| 3 | `test3.py` | 展开手写 `tools` 参数 | 模型 | 对照 schema 与实际函数 |
| 4 | `test4.py` | `bind_tools()` | 模型 | 手写循环 + `ToolMessage` |
| 5 | `test5.py` | 经典 Agent 工具接口 | 模型 | `AgentExecutor` |
| 6 | `test6.py` | `create_agent` 的工具接口 | 模型 | LangGraph 图 + checkpointer |

**学习顺序保持 0～7 节**。新增 3.1 节讲输入与返回值，4.1～4.3 节验证调用顺序、失败处理和真实模型表现，6.1～6.2 节验证记忆隔离与落盘。

**证据边界**：第 1～6.2 节的天气均为模拟数据；第 6.3 节接入真实和风天气每日预报。离线脚本响应只验证执行器；真实天气请求和真实模型调用分别由独立开关开启，观察实际消息轨迹。


## 前情衔接：week23 的多轮对话 → week24 的工具调用

week23 关注历史如何带入下一轮；week24 增加了工具请求与执行结果。两者都围绕消息列表组织上下文，但工具消息还带有需要维护的协议关系。

### 一、两种角色之外，增加一种调用状态和一种结果消息

| 消息 | 含义 |
|---|---|
| `HumanMessage` | 用户输入 |
| `AIMessage(content=...)` | 模型的正文 |
| `AIMessage(tool_calls=[...])` | 模型提出一个或多个工具请求；也可能同时带正文 |
| `ToolMessage(tool_call_id=...)` | 程序执行后回填的结果，关联某次请求 |

`AIMessage(tool_calls=...)` 仍属于 assistant 角色，并非新增的第五种参与者。

### 二、聊天历史与本轮工作区

| | `chat_history` | `agent_scratchpad` |
|---|---|---|
| 本例作用 | 为后续用户轮次提供上下文 | 为当前任务保存中间调用与结果 |
| 管理者 | 历史存储与包装器 | 经典 Agent 执行器 |
| 保存内容 | 本例保存用户输入与最终回答 | 本轮工具调用和工具结果 |
| 生命周期 | 随会话存储保留 | 每次 Agent 执行时组织 |

这里的跨轮聊天历史是**会话内短期记忆**。它不等同于跨会话的用户画像或长期知识。

### 三、类比有用，但对象并不等价

| week23 概念 | week24 对照 | 边界 |
|---|---|---|
| `session_id` | `thread_id` | 都可用于区分会话；后者标识图执行线程，不是用户身份本身 |
| 聊天历史存储 | checkpointer | 前者主要保存消息；后者保存图状态与检查点 |
| `RunnableWithMessageHistory` | LangGraph 持久化 | 当前本地版本对前者给出弃用提示；第 5 节用于理解旧代码 |
| `PydanticOutputParser` | 原生 `tool_calls` | 前者解析生成后的正文；后者使用专门的调用协议字段 |
| `MessagesPlaceholder` | `agent_scratchpad` | 前者是占位符组件；后者是本例赋予占位符的变量名 |

```mermaid
graph LR
    A["week23：消息历史"] --> B["test1：自定义协议与循环"]
    B --> C["test4：原生调用与结果回填"]
    C --> D["test5：经典 AgentExecutor"]
    D --> E["test6：create_agent 与图状态"]
    A --> F["按会话保存历史"]
    F --> E
```

### 四、历史裁剪必须维护调用关系

```text
AIMessage(tool_calls=[{id: "call_1"}, {id: "call_2"}])
ToolMessage(tool_call_id="call_1")
ToolMessage(tool_call_id="call_2")
AIMessage(content="最终回答")
```

按消息数量硬切，可能留下没有结果的调用，或没有请求的结果。对于本笔记使用的原生工具协议，下一次请求前应确保每个调用都有对应结果，且不出现孤立结果。

**最容易实现的保守方案**是按完整用户轮次裁剪；也可以使用能保持工具消息组完整性的裁剪或摘要策略。不能把“必须保留配对关系”误解为所有场景都只能按完整轮次裁剪。


## 0. 环境与运行方式

本地既有环境：Python 3.11.9，langchain 1.4.7、langchain-core 1.7.0、langchain-openai 1.7.3、langchain-classic 1.0.8、langgraph 1.2.18、pydantic 2.14.1。实际版本以下一格输出为准；无需为阅读本笔记盲目升级。

- **基础依赖**：离线实验也需要 Pydantic、langchain-core、LangGraph；经典 Agent 部分另外依赖 langchain-classic。
- **模型配置**：联网部分继续复用 `week23/20261027/lab_config.py` 与 `week23/.env`，不在笔记中填写密钥。模型名以当前配置为准。
- **默认离线运行**：下一格的 `RUN_LIVE_DEMOS = False`。先从头运行，检查协议、工具、错误处理与状态存储；改为 `True` 后重新从第 0 节运行，才会发出真实模型请求并产生 API 用量。
- **额外联网练习**：4.3 节还有独立开关，避免一次执行全部练习。
- **思考模式**：本例沿用配置中的关闭思考选项，方便观察工具轨迹与控制输出预算。思考模式是否有益取决于任务与模型；不应概括为无用消耗。

初始化分别记录 `CONFIG_OK` 与 `LLM_OK`。缺少课程配置或服务不通时，会跳过联网演示；基础包缺失时需要先在当前内核对应环境中安装，不能靠网络开关补足依赖。

第 6.3 节还需要 `requests`、`PyJWT[crypto]`，且 Notebook 内核应使用已有的 `langchain-learning` / week23 `.venv` 环境。缺少依赖时在该内核执行 `%pip install requests "PyJWT[crypto]"`；天气凭据继续读取旁边 `天气预报/config.json`，不写进 Notebook。


In [55]:
import json
import os
import sys
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

RUN_LIVE_DEMOS = False  # 改为 True 才会连接模型；离线执行不加载 .env
if os.environ.get("TOOL_NOTEBOOK_OFFLINE") == "1":
    RUN_LIVE_DEMOS = False  # 自动检查时可强制离线
CONFIG_OK = False
LLM_OK = False
config_error = None


class OfflineModel:
    """离线占位对象：误用时立刻给出可操作的提示，而不是 NoneType / AttributeError。"""

    _hint = ("当前离线：请把 RUN_LIVE_DEMOS 改为 True（并确认 week23/.env 可用），"
             "再从第 0 节重新运行本笔记本。")

    def __getattr__(self, name):
        if name.startswith("__"):
            raise AttributeError(name)
        raise RuntimeError(f"{self._hint}（被访问的成员：{name}）")

    def __repr__(self):
        return "<OfflineModel：未连接任何模型>"


model = OfflineModel()   # 只有联网初始化成功才会被真实模型替换
model_with_tools = None

for pkg in ["langchain", "langchain-core", "langchain-openai",
            "langchain-classic", "langgraph", "pydantic", "python-dotenv"]:
    try:
        print(f"{pkg:18s} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:18s} (未安装)")


def find_lab_dir(start: Path) -> Path:
    """向上寻找既有课程配置；不依赖写死的 D 盘路径。"""
    for base in [start, *start.parents]:
        for candidate in (base / "week23" / "20261027", base / "20261027"):
            if (candidate / "lab_config.py").is_file():
                return candidate
    raise FileNotFoundError("没找到 week23/20261027/lab_config.py；请从 step1 目录树运行")


if not RUN_LIVE_DEMOS:
    print("\n离线模式：不读取模型配置，不请求 API。")
else:
    try:
        lab_dir = str(find_lab_dir(Path.cwd()))
        if lab_dir not in sys.path:
            sys.path.insert(0, lab_dir)
        from lab_config import get_model
        candidate_model = get_model(temperature=0)
        CONFIG_OK = True
        probe = candidate_model.invoke("只回复两个字：你好")
        if not probe.content:
            raise ValueError("连通性响应正文为空，请检查模型、思考模式与输出预算")
        model = candidate_model   # 连通后才替换占位对象
        LLM_OK = True
        print("\n连通性测试 →", repr(str(probe.content)[:40]))
    except Exception as exc:
        config_error = type(exc).__name__
        print("\n初始化或请求失败：", config_error)
        print("检查课程配置、依赖、网络与服务端设置；后续联网演示跳过。")

print(f"\nCONFIG_OK={CONFIG_OK}, LLM_OK={LLM_OK}, model={model!r}")

langchain          1.4.7
langchain-core     1.7.0
langchain-openai   1.7.3
langchain-classic  1.0.8
langgraph          1.2.18
pydantic           2.14.1
python-dotenv      1.2.4

离线模式：不读取模型配置，不请求 API。

CONFIG_OK=False, LLM_OK=False, model=<OfflineModel：未连接任何模型>


## 1. 工具函数由程序执行

在这里的本地函数调用方案中，模型生成工具名与参数；Python 程序负责校验、分发、执行与回填。模型提出调用，不会直接访问你进程中的函数。

$$
\text{模型响应：工具名与参数}
\longrightarrow \text{执行器校验并查表}
\longrightarrow \text{Python 函数执行}
\longrightarrow \text{结果消息}
$$

下面先使用普通 Python 函数，之后才引入工具装饰器。

**模拟数据约定**：此处天气固定返回晴、25°C；后面的结构化天气工具使用一个本地小字典。两者都没有连接天气服务，返回值不能当作真实天气。

**这一格的函数不做任何校验**：`get_weather` 不检查城市是否存在、也不检查参数类型，传任何字符串都返回晴天。校验缺失正是第 3 节引入 `args_schema` 与城市白名单的原因——先看到"没有校验"的样子，才知道 schema 在补什么。

In [39]:
def get_weather(city: str) -> str:
    """返回固定的模拟天气，用于演示工具调用。"""
    return f"[模拟数据] {city}：晴，25°C"


def add(a: float, b: float) -> str:
    """计算两个数的和。"""
    return str(a + b)


TOOLS = {"get_weather": get_weather, "add": add}
print(TOOLS["add"](2, 3))
print(TOOLS["get_weather"]("北京"))
print("docstring 可被用于工具说明；普通函数不会自动把说明发给模型：", get_weather.__doc__)


5
[模拟数据] 北京：晴，25°C
docstring 可被用于工具说明；普通函数不会自动把说明发给模型： 返回固定的模拟天气，用于演示工具调用。


## 2. `test1.py`：用正文 JSON 定义自有协议

先用 Pydantic 描述 `tool_name / tool_args / output`，再把 parser 的格式说明放入 system prompt。模型生成正文后，parser 将其解析成对象。

- `output is None`：申请执行工具。
- `output` 是非空字符串：最终回答；这时约定工具名为空、参数为空字典。

这是自定义的二选一协议，不是原生 function calling。下面先离线观察格式说明和解析，再看完整消息如何回填。

参数目前用通用 `dict` 表示，所以外层 JSON 合法不代表工具名和工具参数正确。第 3、4 节会补上逐工具的 schema 与执行校验。


In [40]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers.pydantic import PydanticOutputParser


class ToolCall(BaseModel):
    tool_name: str = Field(description="工具名")
    tool_args: dict = Field(description="工具参数")
    output: str | None = Field(
        description="不需要工具时写给人的回复；需要调用工具时填 null"
    )


parser = PydanticOutputParser(pydantic_object=ToolCall)
print("========== 塞进 system prompt 的格式说明 ==========")
print(parser.get_format_instructions())

========== 塞进 system prompt 的格式说明 ==========
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"tool_name": {"description": "工具名", "title": "Tool Name", "type": "string"}, "tool_args": {"additionalProperties": true, "description": "工具参数", "title": "Tool Args", "type": "object"}, "output": {"anyOf": [{"type": "string"}, {"type": "null"}], "description": "不需要工具时写给人的回复；需要调用工具时填 null", "title": "Output"}}, "required": ["tool_name", "tool_args", "output"]}
```


In [41]:
# 假装这是模型返回的文本（真实返回值来自 model），看看 parser 能不能吃下
fake_reply = """```json
{"tool_name": "get_weather", "tool_args": {"city": "北京"}, "output": null}
```"""

call = parser.parse(fake_reply)
print(call)
print("工具名 =", call.tool_name, "| 参数 =", call.tool_args, "| 直接回答 =", call.output)

# output 不是 None 的那一支：模型选择"不调工具，直接说"
direct = parser.parse('{"tool_name": "", "tool_args": {}, "output": "你好，我是助手"}')
print("\n直接回答分支 →", direct.output)

# 拿到结构化对象之后，真正执行工具的就是下面这一行
if call.output is None:
    print("\n执行结果 →", TOOLS[call.tool_name](**call.tool_args))

tool_name='get_weather' tool_args={'city': '北京'} output=None
工具名 = get_weather | 参数 = {'city': '北京'} | 直接回答 = None

直接回答分支 → 你好，我是助手

执行结果 → [模拟数据] 北京：晴，25°C


In [42]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "你使用自定义 JSON 协议，所有响应（包括最终回答）都必须满足：{tool_call_instructions}\n"
     "可用工具：get_weather(city: str) 返回模拟天气；add(a: float, b: float) 做加法。"
     "需要工具时 output 为 null；最终回答时 output 填回答，tool_name 填空字符串，tool_args 填空字典。"
     "若已有足够工具结果就回答，不重复查询；明确说明天气是模拟数据。"),
    MessagesPlaceholder("message_history"),
]).partial(tool_call_instructions=parser.get_format_instructions())

raw_chain = prompt | model if LLM_OK else None
history = []


def chat(text: str, max_steps: int = 4, *, chain=None) -> dict:
    """自定义协议：返回 answer / status / model_calls / messages，与 run_tool_loop 的返回契约对齐。"""
    runner = raw_chain if chain is None else chain
    if runner is None:
        raise RuntimeError("没有可用模型；请启用联网，或通过 chain= 传入离线响应器")
    if max_steps < 1:
        raise ValueError("max_steps 必须至少为 1")

    history.append(HumanMessage(content=text))  # 原问题只追加一次，之后一直保留

    def finish(answer, status, count):
        return {"answer": answer, "status": status,
                "model_calls": count, "messages": list(history)}

    for step in range(1, max_steps + 1):
        msg = runner.invoke({"message_history": list(history)})
        raw = msg.content
        history.append(AIMessage(content=raw))  # 保存实际生成的 JSON
        try:
            res = parser.parse(raw)
        except Exception as exc:
            return finish(f"自定义协议解析失败：{type(exc).__name__}；原始输出已保存在 messages",
                          "parse_error", step)

        if res.output is not None:
            if not res.output.strip() or res.tool_name or res.tool_args:
                return finish("自定义协议字段冲突或最终回答为空", "protocol_error", step)
            return finish(res.output, "finished", step)

        try:
            if res.tool_name not in TOOLS:
                raise ValueError(f"未知工具：{res.tool_name}")
            result = TOOLS[res.tool_name](**res.tool_args)
        except (TypeError, ValueError) as exc:
            result = f"工具执行失败：{type(exc).__name__}；请检查工具名和参数"
        print(f"[step {step}] {res.tool_name}({res.tool_args}) → {result}")
        # 仅用于自定义协议教学：结果暂借 human 角色；原生协议在第 4 节改用 ToolMessage。
        history.append(HumanMessage(content=f"程序返回的工具结果（不是新的用户请求）：{result}"))

    return finish("达到模型调用次数上限；messages 保留了原问题和全部中间轨迹",
                  "max_steps", max_steps)


if LLM_OK:
    try:
        live = chat("北京天气怎么样？")
        print("助手：", live["answer"], "| status =", live["status"])
        for message in live["messages"]:
            print(f"{message.type:8s} {message.content}")
    except Exception as exc:
        print("联网演示未完成：", type(exc).__name__)
else:
    print("跳过联网 chat；函数仍可传入离线响应器验证。")

跳过联网 chat；函数仍可传入离线响应器验证。


### 自定义协议的局限，与原生工具调用的边界

| 风险 | 为什么会发生 | 可补充的机制 |
|---|---|---|
| 正文不是预期 JSON | 格式说明无法代替生成约束 | 在服务支持时使用结构化输出、解析校验和有限重试 |
| JSON 合法但字段语义矛盾 | 类型正确不等于“调用/回答”分支正确 | 互斥字段校验，或用判别联合分别定义两种响应 |
| 缺参、错选工具、编造参数 | 工具选择与参数来源仍需模型判断 | 逐工具校验；信息不足时追问 |
| 已有结果却重复调用 | 决策、上下文或停止策略不充分 | 明确任务状态、限制步数，必要时检测重复请求 |

原生 `tool_calls` 把调用与正文分开，并提供用于关联结果的 id；它不自动保证业务正确。
`tool_call_id` **负责关联，不负责防重复**。`bind_tools` **绑定工具描述，不执行工具**。

本笔记不把未保留日志的历史现象当作本次实验结论。实际模型是否失败，以开启联网演示后的输出为准。


## 3. `test2.py` / `test3.py`：绑定原生工具 schema

`@tool` 将函数包装成 LangChain 工具；`bind_tools` 将工具名称、描述和参数 schema 绑定到模型请求。
随后模型响应可带有 `tool_calls`，通常包含 `name / args / id`，正文 `content` 可以为空，也可以与调用同时存在。

- 不指定 `args_schema` 时，工具通常根据函数参数类型推导输入 schema。
- 指定 `args_schema=WeatherArgs` 时，显式 Pydantic 模型描述工具输入。
- schema 描述**传入函数的参数**；返回值是执行后的观察结果。

原生调用依赖模型和服务端支持；使用兼容接口并不保证每个供应商支持完全相同的约束或参数。程序仍需校验和处理失败。


In [43]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field, ConfigDict


class WeatherArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    city: str = Field(min_length=1, description="城市名，如北京、上海、成都", examples=["北京"])
    # examples 是推荐写法；旧 example= 在当前 Pydantic v2 会警告，并不一定被忽略。


MOCK_WEATHER = {
    "北京": {"weather": "晴", "temperature_c": 25.0},
    "上海": {"weather": "多云", "temperature_c": 22.0},
    "成都": {"weather": "阴", "temperature_c": 20.0},
}


@tool(args_schema=WeatherArgs)
def query_weather(city: str) -> str:
    """查询北京、上海或成都的模拟天气；不提供实时天气。"""
    if city not in MOCK_WEATHER:
        raise ValueError("模拟数据仅覆盖北京、上海、成都；当前城市没有记录")
    return json.dumps({"city": city, **MOCK_WEATHER[city], "source": "mock"}, ensure_ascii=False)


print("name        :", query_weather.name)
print("description :", query_weather.description)
print("args        :", query_weather.args)
print("\n输入 JSON Schema：")
print(json.dumps(query_weather.args_schema.model_json_schema(), ensure_ascii=False, indent=2))
print("\n执行后才得到返回值：", query_weather.invoke({"city": "北京"}))


name        : query_weather
description : 查询北京、上海或成都的模拟天气；不提供实时天气。
args        : {'city': {'description': '城市名，如北京、上海、成都', 'minLength': 1, 'title': 'City', 'type': 'string'}}

输入 JSON Schema：
{
  "additionalProperties": false,
  "properties": {
    "city": {
      "description": "城市名，如北京、上海、成都",
      "examples": [
        "北京"
      ],
      "minLength": 1,
      "title": "City",
      "type": "string"
    }
  },
  "required": [
    "city"
  ],
  "title": "WeatherArgs",
  "type": "object"
}

执行后才得到返回值： {"city": "北京", "weather": "晴", "temperature_c": 25.0, "source": "mock"}


In [44]:
# 即使离线，也能查看转换后的工具 schema。
from langchain_core.utils.function_calling import convert_to_openai_tool

converted_tool = convert_to_openai_tool(query_weather)
print(json.dumps(converted_tool, ensure_ascii=False, indent=2))

if LLM_OK:
    try:
        model_with_tools = model.bind_tools([query_weather])
        res = model_with_tools.invoke("用模拟数据查询北京天气")
        print("\n正文 content：", repr(res.content))
        print("结构化 tool_calls：", res.tool_calls)
        print("解析失败的调用：", res.invalid_tool_calls)
    except Exception as exc:
        model_with_tools = None
        print("绑定或请求失败：", type(exc).__name__)
else:
    print("\n离线：仅展示 schema，不发出模型请求。")


{
  "type": "function",
  "function": {
    "name": "query_weather",
    "description": "查询北京、上海或成都的模拟天气；不提供实时天气。",
    "parameters": {
      "properties": {
        "city": {
          "description": "城市名，如北京、上海、成都",
          "minLength": 1,
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    }
  }
}

离线：仅展示 schema，不发出模型请求。


In [45]:
# 手写“当前笔记修正后”的同义 schema；不是原始 test3.py 的原样复制。
TOOLS_JSON = [{
    "type": "function",
    "function": {
        "name": "query_weather",
        "description": query_weather.description,
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "minLength": 1,
                         "description": "城市名，如北京、上海、成都"}
            },
            "required": ["city"],
        },
    },
}]

generated = converted_tool["function"]["parameters"]
written = TOOLS_JSON[0]["function"]["parameters"]
assert generated["required"] == written["required"] == ["city"]
assert set(generated["properties"]) == set(written["properties"]) == {"city"}
assert generated["properties"]["city"]["minLength"] == written["properties"]["city"]["minLength"] == 1
print("请求 schema 的字段、必填项和最小长度一致。")
print("注意：当前转换器不保留全部 Pydantic 配置；extra=forbid 仍由本地工具校验。")

if LLM_OK:
    try:
        res2 = model.invoke("用模拟数据查询北京天气", tools=TOOLS_JSON)
        print("手写 tools 的响应 →", res2.tool_calls)
    except Exception as exc:
        print("手写 schema 联网演示未完成：", type(exc).__name__)
else:
    print("离线：不发出模型请求。")


请求 schema 的字段、必填项和最小长度一致。
注意：当前转换器不保留全部 Pydantic 配置；extra=forbid 仍由本地工具校验。
离线：不发出模型请求。


### 3.1 工具输入、返回值、schema：不要让模型填写查询答案

原始 `test3.py` 的 `Weather` 把 `city / weather / temperature` 都声明为必填输入，同时函数把这三个参数直接拼成返回字符串。
这会让模型提供本来应该由天气工具查询得到的天气和温度。更合理的边界是：

```text
输入 WeatherArgs：city
        ↓ 程序查询外部服务或本例的模拟字典
结果：city / weather / temperature_c / source
```

原脚本还有一处不一致：Pydantic schema 的 required 是三个字段，手写 tools 却只有 city。本笔记统一为只输入 city，因而不声称原脚本两份 schema 本来就完全等价。

另外，`Field(example=...)` 在当前 Pydantic v2 中属于弃用的额外参数写法，可能仍进入 JSON Schema；推荐 `examples=[...]`，不是因为旧写法一定被忽略。
本地 Pydantic 模型、转换后的请求 schema 和服务端实际采用的约束也可能不同：当前转换器会省略部分配置，所以本地参数校验仍然需要保留。

下面直接检验缺少城市、空城市和额外参数是否被拒绝。工具输入校验通过后，城市是否有记录仍属于工具执行逻辑。


In [46]:
for bad_args in ({}, {"city": "  "}, {"city": "北京", "temperature_c": 99}):
    try:
        WeatherArgs.model_validate(bad_args)
    except Exception as exc:
        print("拒绝输入：", bad_args, "→", type(exc).__name__)
    else:
        raise AssertionError("不应接受该输入")

data = json.loads(query_weather.invoke({"city": "北京"}))
assert data["temperature_c"] == 25.0 and data["source"] == "mock"
print("返回值来自工具：", data)


拒绝输入： {} → ValidationError
拒绝输入： {'city': '  '} → ValidationError
拒绝输入： {'city': '北京', 'temperature_c': 99} → ValidationError
返回值来自工具： {'city': '北京', 'weather': '晴', 'temperature_c': 25.0, 'source': 'mock'}


## 4. `test4.py`：执行、回填、停止

原生调用中，每个 `ToolMessage.tool_call_id` 对应某个 `AIMessage.tool_calls[i].id`。一条 assistant 消息提出两个调用，就应在继续请求模型前补齐两个结果。

```text
HumanMessage("分别查询北京和上海天气")
AIMessage(tool_calls=[{id: "call_1", ...}, {id: "call_2", ...}])
ToolMessage(tool_call_id="call_1", content="...")
ToolMessage(tool_call_id="call_2", content="...")
AIMessage(content="最终回答")
```

`test4.py` 中的 `if input == "exit"` 比较了内置函数，退出判断应使用实际输入变量；`get_session_history` 应返回 `store[session_id]`，而非整个字典。

下面将最小循环补成可观察的教学执行器：

1. 保留用户原问题与模型原始响应。
2. 按白名单选择工具，使用工具自身 schema 校验参数。
3. 把可处理的工具失败作为错误结果回填；未知程序错误继续抛出，供开发者排查。
4. 每次模型响应可有多个调用；本例逐个执行，并不真正并发。
5. 返回 `answer / status / messages / model_calls`，区分最终答案与程序停止提示。

`max_steps` 是**模型请求次数上限**。一次工具调用后还需一次模型请求生成最终回答，因此一般至少需要两步；工具函数阻塞的超时需要在工具内部或执行层另外设置，步数上限不能替代超时。


In [56]:
from langchain_core.messages import SystemMessage, ToolMessage
from langchain_core.tools import ToolException
from pydantic import ValidationError

TOOL_BY_NAME = {query_weather.name: query_weather}


def check_tool_pairs(messages):
    """检查已完成的消息列表：无重复 id、无孤立结果、无待回填调用。

    「重复」既包括同一批内重复，也包括跨批次复用同一个 id——
    后者会让历史里出现两条 ToolMessage 指向同一次调用，模型无法区分。
    """
    pending = set()
    seen = set()
    for message in messages:
        if isinstance(message, ToolMessage):
            if message.tool_call_id not in pending:
                raise ValueError("出现孤立或重复的工具结果")
            pending.remove(message.tool_call_id)
        else:
            if pending:
                raise ValueError("工具结果未补齐就出现下一条非工具消息")
            calls = getattr(message, "tool_calls", []) or []
            ids = [call.get("id") for call in calls]
            if any(not cid for cid in ids) or len(set(ids)) != len(ids):
                raise ValueError("调用 id 缺失或同一批内重复")
            reused = set(ids) & seen
            if reused:
                raise ValueError(f"调用 id 跨批次重复：{sorted(reused)}")
            seen.update(ids)
            pending.update(ids)
    if pending:
        raise ValueError("仍有调用没有工具结果")
    return True


def execute_call(tc, tools_by_name):
    """把预期内的失败变成 ToolMessage；不吞掉任意程序异常。"""
    selected = tools_by_name.get(tc["name"])
    error = None
    if selected is None:
        error = {"type": "UnknownTool", "message": "未知工具；可用名称：" + ", ".join(tools_by_name)}
    else:
        try:
            result = selected.invoke(tc["args"])
        except ValidationError as exc:
            error = {"type": "ValidationError",
                     "fields": [{"loc": list(e["loc"]), "type": e["type"]}
                                for e in exc.errors(include_input=False, include_url=False)]}
        except (ToolException, TimeoutError, ConnectionError, ValueError) as exc:
            error = {"type": type(exc).__name__, "message": str(exc)}
    if error is not None:
        return ToolMessage(
            content=json.dumps({"ok": False, "error": error}, ensure_ascii=False),
            tool_call_id=tc["id"], name=tc["name"], status="error",
        )
    content = result if isinstance(result, str) else json.dumps(result, ensure_ascii=False)
    return ToolMessage(content=content, tool_call_id=tc["id"], name=tc["name"])


def run_tool_loop(question: str, max_steps: int = 4, verbose: bool = True,
                  *, bound_model=None, tools_by_name=None, system_prompt=None):
    """返回完整轨迹；bound_model 可传入离线响应器，验证同一套执行代码。"""
    runner = model_with_tools if bound_model is None else bound_model
    registry = TOOL_BY_NAME if tools_by_name is None else tools_by_name
    if runner is None:
        raise RuntimeError("没有可用模型；请启用联网或传入离线响应器")
    if max_steps < 1:
        raise ValueError("max_steps 必须至少为 1")

    messages = [
        SystemMessage(content=(system_prompt if system_prompt is not None else
                               "你可以使用工具。信息不足先追问；如工具失败请如实说明。天气均为模拟数据。")),
        HumanMessage(content=question),
    ]

    def finish(answer, status, count):
        return {"answer": answer, "status": status,
                "model_calls": count, "messages": messages}

    for step in range(1, max_steps + 1):
        check_tool_pairs(messages)
        ai = runner.invoke(messages)
        messages.append(ai)
        if getattr(ai, "invalid_tool_calls", []):
            return finish("调用参数未能解析；请查看轨迹并修复协议", "invalid_tool_calls", step)
        if not ai.tool_calls:
            if not ai.content:
                return finish("模型未返回正文或调用", "empty_response", step)
            return finish(ai.content, "finished", step)

        ids = [tc.get("id") for tc in ai.tool_calls]
        if any(not cid for cid in ids) or len(set(ids)) != len(ids):
            return finish("调用 id 缺失或重复，未执行本批工具", "protocol_error", step)
        for tc in ai.tool_calls:
            observation = execute_call(tc, registry)
            messages.append(observation)
            if verbose:
                print(f"[model {step}] {tc['name']}({tc['args']}) id={tc['id']}")
                print("  →", observation.status, observation.content)
        check_tool_pairs(messages)

    return finish("达到模型调用次数上限，尚未生成最终答案", "max_steps", max_steps)


if LLM_OK and model_with_tools is not None:
    try:
        demo = run_tool_loop("用模拟数据查询北京天气")
        print("状态：", demo["status"], "| 回答：", demo["answer"])
    except Exception as exc:
        print("联网执行器演示未完成：", type(exc).__name__)
else:
    print("离线：执行器已定义；下一节用固定响应验证。")

离线：执行器已定义；下一节用固定响应验证。


### 4.1 多工具调用：同一批独立请求，与跨轮依赖

**独立请求**：“查询北京和上海天气”。模型可以一次提出两条调用，执行器补齐两条结果再请求总结。模型也可能选择逐轮调用；一次返回两条不是所有模型都保证的行为。

**依赖请求**：“北京气温加 5 度是多少”。先查询温度，回填后模型才能用该温度构造加法参数。不能把尚未取得的温度当作已知值填写。

```mermaid
graph LR
    A["独立：北京与上海天气"] --> B["可在同一响应提出两次查询"]
    B --> C["分别执行并回填两个 id"]
    C --> D["模型总结"]
    E["依赖：北京温度加 5"] --> F["先查询北京"]
    F --> G["回填 25°C（模拟）"]
    G --> H["再调用 add_numbers(25, 5)"]
    H --> I["回填 30，再总结"]
```

下面的 ScriptedModel 按预先给定的响应运行，不使用网络。依赖场景的加法参数从上一条 ToolMessage 实际读取，而不是提前写死。


In [61]:
class AddArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", allow_inf_nan=False)
    a: float = Field(description="第一个加数")
    b: float = Field(description="第二个加数")


@tool(args_schema=AddArgs)
def add_numbers(a: float, b: float) -> str:
    """计算两个数的和。"""
    return str(a + b)


ALL_TOOLS = {t.name: t for t in [query_weather, add_numbers]}


class ScriptedModel:
    """固定响应或回调，用于验证程序分支；不是语言模型能力测试。

    除了返回固定响应，它还会：
    1. 校验执行器确实传入了非空历史，且末条是用户或工具消息；
    2. 记录每次请求的消息快照（self.calls），供断言"执行器发了什么"。
    """

    def __init__(self, replies):
        self.replies = list(replies)
        self.index = 0
        self.calls = []

    @staticmethod
    def _history(messages):
        """兼容两种调用方式：执行器传消息列表，自定义协议传 {"message_history": [...]}。"""
        if isinstance(messages, dict):
            return list(messages.get("message_history") or [])
        return list(messages)

    def invoke(self, messages):
        if self.index >= len(self.replies):
            raise AssertionError("执行器发出了预期外的模型请求")
        snapshot = self._history(messages)
        if not snapshot:
            raise AssertionError("执行器必须传入非空的消息历史")
        if not isinstance(snapshot[-1], (HumanMessage, ToolMessage)):
            raise AssertionError(
                f"本次请求的末条消息应为用户或工具消息，实际为 {type(snapshot[-1]).__name__}"
            )
        self.calls.append(snapshot)
        reply = self.replies[self.index]
        self.index += 1
        reply = reply(messages) if callable(reply) else reply
        return reply.model_copy(deep=True)


def request_tool(name, args, cid):
    return AIMessage(content="", tool_calls=[
        {"name": name, "args": args, "id": cid, "type": "tool_call"}
    ])


def add_after_weather(messages):
    observation = json.loads(messages[-1].content)
    assert messages[-1].name == "query_weather" and observation["source"] == "mock"
    return request_tool("add_numbers", {"a": observation["temperature_c"], "b": 5}, "serial_add")


def answer_after_add(messages):
    value = float(messages[-1].content)
    return AIMessage(content=f"模拟温度加 5 后为 {value:g}°C。")


parallel_ai = AIMessage(content="", tool_calls=[
    {"name": "query_weather", "args": {"city": "北京"}, "id": "parallel_bj", "type": "tool_call"},
    {"name": "query_weather", "args": {"city": "上海"}, "id": "parallel_sh", "type": "tool_call"},
])
parallel_script = ScriptedModel([parallel_ai, AIMessage(content="两地模拟天气查询完成。")])
serial_script = ScriptedModel([
    request_tool("query_weather", {"city": "北京"}, "serial_weather"),
    add_after_weather, answer_after_add,
])

parallel_demo = run_tool_loop(
    "分别查询北京和上海的模拟天气", verbose=False, tools_by_name=ALL_TOOLS,
    bound_model=parallel_script,
)
serial_demo = run_tool_loop(
    "北京气温加 5 度是多少", verbose=False, tools_by_name=ALL_TOOLS,
    bound_model=serial_script,
)
for label, result in [("同一批独立调用", parallel_demo), ("跨轮依赖调用", serial_demo)]:
    print("\n", label, "| model_calls =", result["model_calls"])
    for message in result["messages"]:
        print(message.type, message.content, getattr(message, "tool_calls", "") or "")
assert parallel_demo["model_calls"] == 2
assert serial_demo["model_calls"] == 3 and "30°C" in serial_demo["answer"]
# 响应器记录到的请求数，应与执行器上报的模型调用次数一致
assert len(parallel_script.calls) == 2 and len(serial_script.calls) == 3
# 依赖链最后一次请求的末条消息，应是 add_numbers 回填的工具结果
assert isinstance(serial_script.calls[-1][-1], ToolMessage)
assert serial_script.calls[-1][-1].content == "30.0"
print("\n响应器收到的请求数与末条消息类型：通过")


 同一批独立调用 | model_calls = 2
system 你可以使用工具。信息不足先追问；如工具失败请如实说明。天气均为模拟数据。 
human 分别查询北京和上海的模拟天气 
ai  [{'name': 'query_weather', 'args': {'city': '北京'}, 'id': 'parallel_bj', 'type': 'tool_call'}, {'name': 'query_weather', 'args': {'city': '上海'}, 'id': 'parallel_sh', 'type': 'tool_call'}]
tool {"city": "北京", "weather": "晴", "temperature_c": 25.0, "source": "mock"} 
tool {"city": "上海", "weather": "多云", "temperature_c": 22.0, "source": "mock"} 
ai 两地模拟天气查询完成。 

 跨轮依赖调用 | model_calls = 3
system 你可以使用工具。信息不足先追问；如工具失败请如实说明。天气均为模拟数据。 
human 北京气温加 5 度是多少 
ai  [{'name': 'query_weather', 'args': {'city': '北京'}, 'id': 'serial_weather', 'type': 'tool_call'}]
tool {"city": "北京", "weather": "晴", "temperature_c": 25.0, "source": "mock"} 
ai  [{'name': 'add_numbers', 'args': {'a': 25.0, 'b': 5}, 'id': 'serial_add', 'type': 'tool_call'}]
tool 30.0 
ai 模拟温度加 5 后为 30°C。 

响应器收到的请求数与末条消息类型：通过


### 4.2 错误、追问与停止：离线验证程序行为

下面覆盖不调用工具、缺少信息先追问、参数缺失、未知工具、查询失败、调用次数耗尽、无法解析的调用以及空响应。

- 合法调用执行失败时，仍需回填对应 id 的错误结果；不能伪造成功。
- 已解析的调用缺少参数，由工具 schema 拒绝；无法解析参数的调用在 `invalid_tool_calls` 中单独处理。
- 达到上限时，返回 `status="max_steps"`，轨迹里仍保留已取得的工具结果。停止提示来自程序，不冒充模型的最终答案。
- 对于格式损坏的调用，本例停止并保留轨迹，没有实现自动修复；不要把该轨迹直接续接到下一次原生 API 请求。

这些断言验证的是程序对给定响应的处理。真实模型是否会追问、改参或忠于结果，需要 4.3 节另外观察。


In [62]:
offline_reports = []


def verify_case(label, replies, expected_status, *, max_steps=4):
    result = run_tool_loop(
        label, verbose=False, max_steps=max_steps,
        bound_model=ScriptedModel(replies), tools_by_name=ALL_TOOLS,
    )
    assert result["status"] == expected_status, (label, result["status"])
    if expected_status not in {"invalid_tool_calls", "protocol_error"}:
        assert check_tool_pairs(result["messages"])
    offline_reports.append((label, result["status"], result["model_calls"]))
    return result


direct_check = verify_case("你好", [AIMessage(content="你好！")], "finished")
clarify_check = verify_case("查天气", [AIMessage(content="请问查询哪个城市？")], "finished")
assert not any(isinstance(m, ToolMessage) for m in direct_check["messages"])
assert not any(isinstance(m, ToolMessage) for m in clarify_check["messages"])

repair_check = verify_case("缺参后修正", [
    request_tool("query_weather", {}, "missing_city"),
    request_tool("query_weather", {"city": "北京"}, "fixed_city"),
    AIMessage(content="模拟天气已查到。"),
], "finished")
assert [m.status for m in repair_check["messages"] if isinstance(m, ToolMessage)] == ["error", "success"]

unknown_check = verify_case("未知工具", [
    request_tool("unknown_weather", {"city": "北京"}, "unknown"),
    AIMessage(content="该工具不可用，无法完成查询。"),
], "finished")
assert "UnknownTool" in unknown_check["messages"][3].content

failure_check = verify_case("模拟城市无记录", [
    request_tool("query_weather", {"city": "不存在的城市"}, "no_record"),
    AIMessage(content="模拟数据中没有该城市，无法给出天气。"),
], "finished")
assert failure_check["messages"][3].status == "error"

limit_check = verify_case("达到一次模型调用上限", [
    request_tool("query_weather", {"city": "北京"}, "budget"),
], "max_steps", max_steps=1)
assert isinstance(limit_check["messages"][-1], ToolMessage)

invalid_ai = AIMessage(content="", invalid_tool_calls=[
    {"name": "query_weather", "args": "{", "id": "broken",
     "error": "invalid JSON", "type": "invalid_tool_call"}
])
verify_case("无法解析的调用", [invalid_ai], "invalid_tool_calls")
verify_case("空响应", [AIMessage(content="")], "empty_response")

duplicate_ai = AIMessage(content="", tool_calls=[
    {"name": "query_weather", "args": {"city": "北京"}, "id": "same", "type": "tool_call"},
    {"name": "query_weather", "args": {"city": "上海"}, "id": "same", "type": "tool_call"},
])
verify_case("同一批重复 id", [duplicate_ai], "protocol_error")

# 主动破坏配对关系，验证检查器确实能发现问题。
for bad_messages in (
    [request_tool("query_weather", {"city": "北京"}, "orphan_call")],
    [ToolMessage(content="无对应请求", tool_call_id="orphan_result")],
    # 跨批次复用同一个 id：两轮都叫 reused_id，检查器也应拒绝
    [request_tool("query_weather", {"city": "北京"}, "reused_id"),
     ToolMessage(content="第一轮结果", tool_call_id="reused_id"),
     request_tool("query_weather", {"city": "上海"}, "reused_id"),
     ToolMessage(content="第二轮结果", tool_call_id="reused_id")],
):
    try:
        check_tool_pairs(bad_messages)
    except ValueError:
        pass
    else:
        raise AssertionError("应拒绝不完整或 id 重复的调用轨迹")

# 同样验证手写 JSON 协议保存原问题，并能继续下一轮。
history.clear()
manual_runner = ScriptedModel([
    AIMessage(content='{"tool_name":"get_weather","tool_args":{"city":"北京"},"output":null}'),
    AIMessage(content='{"tool_name":"","tool_args":{},"output":"北京模拟温度为25°C。"}'),
])
manual_answer = chat("北京天气怎么样？", chain=manual_runner)
assert isinstance(history[0], HumanMessage) and history[0].content == "北京天气怎么样？"
assert manual_answer["status"] == "finished" and "25" in manual_answer["answer"]
assert len(manual_runner.calls) == manual_answer["model_calls"] == 2

followup_runner = ScriptedModel([
    AIMessage(content='{"tool_name":"","tool_args":{},"output":"你问了北京天气怎么样。"}'),
])
followup = chat("我刚才具体问了什么？", chain=followup_runner)
assert followup["status"] == "finished"
assert any(isinstance(m, HumanMessage) and m.content == "我刚才具体问了什么？" for m in history)
print("\n自定义协议保留原问题与后续问题：通过")
print("两种多工具轨迹与消息配对检查：通过")
for row in offline_reports:
    print(f"{row[0]} → status={row[1]}, model_calls={row[2]}")
history.clear()  # 离线练习结束，不污染后续真实对话

[step 1] get_weather({'city': '北京'}) → [模拟数据] 北京：晴，25°C

自定义协议保留原问题与后续问题：通过
两种多工具轨迹与消息配对检查：通过
你好 → status=finished, model_calls=1
查天气 → status=finished, model_calls=1
缺参后修正 → status=finished, model_calls=3
未知工具 → status=finished, model_calls=2
模拟城市无记录 → status=finished, model_calls=2
达到一次模型调用上限 → status=max_steps, model_calls=1
无法解析的调用 → status=invalid_tool_calls, model_calls=1
空响应 → status=empty_response, model_calls=1
同一批重复 id → status=protocol_error, model_calls=1


### 4.3 真实模型观察：另外开启，记录事实

先在第 0 节启用联网，再将下面的 `RUN_EXTRA_LIVE_DEMOS` 改为 True。这里不会把固定脚本的成功当作真实模型的成绩。

| 问题 | 观察重点 |
|---|---|
| 你好 | 是否无需工具就回答 |
| 帮我查天气 | 没有城市时是否追问 |
| 查询北京模拟天气 | 参数、结果与回答是否一致，是否说明模拟数据 |
| 查询北京和上海模拟天气 | 是否提出多个调用；各 id 是否正确对应结果 |
| 北京模拟气温加 5 度，务必用加法工具 | 是否先查询，再基于返回值调用加法 |
| 查询不存在城市的模拟天气 | 是否如实说明无记录，而非编造结果 |

可以手工记录：工具选择是否正确、参数是否有效、调用先后是否合理、结果是否被忠实使用、模型请求次数。即使协议校验通过，最终回答仍可能答错。


In [50]:
RUN_EXTRA_LIVE_DEMOS = False
live_results = []
if RUN_EXTRA_LIVE_DEMOS and LLM_OK:
    try:
        multi_model = model.bind_tools(list(ALL_TOOLS.values()))
        questions = [
            "你好",
            "帮我查天气",
            "查询北京模拟天气",
            "查询北京和上海模拟天气",
            "北京模拟气温加 5 度，务必使用加法工具",
            "查询不存在城市的模拟天气",
        ]
        for question in questions:
            result = run_tool_loop(
                question, max_steps=5, bound_model=multi_model,
                tools_by_name=ALL_TOOLS,
            )
            live_results.append({"question": question, **result})
            print("\n问题：", question)
            print("状态：", result["status"], "| 回答：", result["answer"])
    except Exception as exc:
        print("额外联网实验未完成：", type(exc).__name__)
else:
    print("额外联网实验未开启。")


额外联网实验未开启。


## 5. `test5.py`：经典 `AgentExecutor` 如何接管循环

| 手写执行器中的工作 | 经典 Agent 中的对应物 |
|---|---|
| 把调用与结果带入下一次请求 | `agent_scratchpad` 与消息格式化 |
| 限制工具决策轮数 | `max_iterations` |
| 按工具名分发 | `tools` 列表 |
| 跨用户轮次保存上下文 | 外包 `RunnableWithMessageHistory` |

prompt 中的 `chat_history` 用于会话内短期历史；`agent_scratchpad` 用于当前任务的中间步骤。本例历史包装器保存用户输入与最终输出，不自动等于完整工具轨迹存档。

**两个边界**：

- `verbose=True` 可打印执行日志，不代表能看到模型内部隐藏思考过程。
- `handle_parsing_errors=True` 处理 Agent 输出解析错误，不负责兜底工具抛出的所有异常。工具错误需要单独处理。

本地现有版本会对历史相关旧接口发出弃用提示。保留这一节是为了读懂旧项目；新代码接着看第 6 节。离线模式下不构造依赖模型的 Agent。


In [51]:
agent_with_history = None
store = {}

if not LLM_OK:
    print("跳过经典 Agent 联网演示。")
else:
    try:
        from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
        from langchain_core.chat_history import InMemoryChatMessageHistory
        from langchain_core.runnables.history import RunnableWithMessageHistory

        agent_prompt = ChatPromptTemplate.from_messages([
            ("system", "你可以使用工具。天气是模拟数据，回答时说明。"),
            MessagesPlaceholder("chat_history"),
            ("human", "{input}"),
            MessagesPlaceholder("agent_scratchpad"),
        ])
        agent = create_tool_calling_agent(model, [query_weather], agent_prompt)
        executor = AgentExecutor(
            agent=agent, tools=[query_weather], verbose=True,
            handle_parsing_errors=True,  # 仅 Agent 输出解析错误
            max_iterations=5,
        )

        def get_session_history(session_id: str):
            if session_id not in store:
                store[session_id] = InMemoryChatMessageHistory()
            return store[session_id]

        agent_with_history = RunnableWithMessageHistory(
            executor, get_session_history,
            input_messages_key="input", output_messages_key="output",
            history_messages_key="chat_history",
        )
        cfg_classic = {"configurable": {"session_id": "classic_demo_001"}}
        r1 = agent_with_history.invoke({"input": "用模拟数据查询北京天气"}, config=cfg_classic)
        print("\n第一轮 →", r1["output"])
        r2 = agent_with_history.invoke({"input": "我刚才问的是哪个城市？"}, config=cfg_classic)
        print("第二轮 →", r2["output"])
    except Exception as exc:
        print("经典 Agent 演示未完成：", type(exc).__name__)


跳过经典 Agent 联网演示。


## 6. `test6.py`：`create_agent` 与图状态

`create_agent` 是本例新建 Agent 的入口，底层使用 LangGraph。它组织模型节点、工具执行和状态更新，支持 checkpointer。

```python
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

graph = create_agent(
    model=model,
    tools=[query_weather],
    system_prompt="你可以使用工具。天气是模拟数据。",
    checkpointer=InMemorySaver(),
)
```

1. 本例使用静态 `system_prompt`；复杂提示与行为还可通过状态、上下文和 middleware 配置。
2. `thread_id` 标识图线程。应用应自行设计用户与线程的映射，不能把它当作登录鉴权机制。
3. 返回状态中的 `messages` 包含当前线程的消息轨迹，可与第 4 节比较。
4. `InMemorySaver` 在当前进程内保存检查点。原脚本使用的 `MemorySaver` 在当前环境中是它的兼容别名；更换名称不会让数据自动落盘。

重复运行会改变已有线程状态。本例每次重建内存 saver，可获得新的演示状态；要检查跨调用记忆，使用同一个图对象、同一个 saver 和同一个 `thread_id`。


In [52]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

graph = None
if not LLM_OK:
    print("跳过真实 Agent；下一节可离线检查状态存储。")
else:
    try:
        graph = create_agent(
            model=model, tools=[query_weather],
            system_prompt="你可以使用工具；信息不足先追问。天气是模拟数据，回答时说明。",
            checkpointer=InMemorySaver(),
        )
        cfg_graph = {"configurable": {"thread_id": "graph_demo_001"},
                     "recursion_limit": 12}  # 图调度上限，与手写 max_steps 不一一对应
        out = graph.invoke(
            {"messages": [{"role": "user", "content": "用模拟数据查询北京天气"}]},
            config=cfg_graph,
        )
        print("---------- 完整消息轨迹 ----------")
        for message in out["messages"]:
            extra = f" tool_calls={message.tool_calls}" if getattr(message, "tool_calls", []) else ""
            print(f"{message.type:8s} {message.content}{extra}")
        out2 = graph.invoke(
            {"messages": [{"role": "user", "content": "我刚才问的是哪个城市？"}]},
            config=cfg_graph,
        )
        print("\n记忆验证 →", out2["messages"][-1].content)
        fresh = graph.get_state({"configurable": {"thread_id": "graph_demo_other"}})
        print("另一个线程的初始消息数：", len(fresh.values.get("messages", [])))
    except Exception as exc:
        print("真实 Agent 演示未完成：", type(exc).__name__)


跳过真实 Agent；下一节可离线检查状态存储。


### 6.1 离线检查记忆：同线程恢复、不同线程隔离

下面使用一个只回固定确认消息的 LangGraph 节点，验证检查点读写。它不调用模型，因此只能证明状态保存与隔离，不能证明模型理解了历史。

同一个用户可以有多个线程；应用根据需求生成线程标识。同一个 saver 下，不同 thread_id 的状态应分开。同线程复用检查点也不代表模型权重发生变化。


In [53]:
from langgraph.graph import StateGraph, MessagesState, START, END


def acknowledge(state):
    return {"messages": [AIMessage(content="离线节点已记录本轮输入。")]}


memory_builder = StateGraph(MessagesState)
memory_builder.add_node("ack", acknowledge)
memory_builder.add_edge(START, "ack")
memory_builder.add_edge("ack", END)
memory_saver = InMemorySaver()
memory_graph = memory_builder.compile(checkpointer=memory_saver)
thread_a = {"configurable": {"thread_id": "offline_thread_a"}}
thread_b = {"configurable": {"thread_id": "offline_thread_b"}}

memory_graph.invoke({"messages": [HumanMessage(content="第一轮：北京")]}, config=thread_a)
remembered = memory_graph.invoke(
    {"messages": [HumanMessage(content="第二轮：继续讨论")]}, config=thread_a,
)
users_a = [m.content for m in remembered["messages"] if isinstance(m, HumanMessage)]
assert users_a == ["第一轮：北京", "第二轮：继续讨论"]
assert not memory_graph.get_state(thread_b).values.get("messages", [])

# 使用同一 saver 重建图，仍能读取；新建内存 saver 则没有原状态。
rebuilt_graph = memory_builder.compile(checkpointer=memory_saver)
assert len(rebuilt_graph.get_state(thread_a).values["messages"]) == 4
fresh_graph = memory_builder.compile(checkpointer=InMemorySaver())
assert not fresh_graph.get_state(thread_a).values.get("messages", [])
print("同线程历史恢复：通过")
print("不同线程状态隔离：通过")
print("同 saver 重建可读、新内存 saver 无历史：通过")


同线程历史恢复：通过
不同线程状态隔离：通过
同 saver 重建可读、新内存 saver 无历史：通过


### 6.2 可选：SQLite 检查点与跨重启恢复

磁盘持久化保存的是线程状态，并不会自动变成跨线程长期记忆。
可选依赖是 `langgraph-checkpoint-sqlite`；需要时在当前内核环境中安装，例如 `%pip install langgraph-checkpoint-sqlite`，安装后按提示重启内核。

下面默认关闭，启用后在当前目录的 `work/tool_calling_checkpoints.sqlite` 写入教学检查点。示例先关闭数据库连接，再重新连接检查记录；这验证磁盘恢复，不等同于已经实测过重启内核。

真正验证跨内核：启用运行一次 → 重启内核 → 从头运行至此且保持同一工作目录、数据库路径与 thread_id。第二次执行应读取已有记录。示例仅在没有状态时写入，避免重跑不断叠加。


In [54]:
RUN_SQLITE_DEMO = False
if not RUN_SQLITE_DEMO:
    print("SQLite 实验未开启；本格未写入数据库。")
else:
    try:
        from langgraph.checkpoint.sqlite import SqliteSaver
    except ImportError:
        print("请先在当前内核中安装 langgraph-checkpoint-sqlite。")
    else:
        db_path = Path.cwd() / "work" / "tool_calling_checkpoints.sqlite"
        db_path.parent.mkdir(parents=True, exist_ok=True)
        sqlite_cfg = {"configurable": {"thread_id": "sqlite_tool_lesson_v1"}}
        with SqliteSaver.from_conn_string(str(db_path)) as saver:
            disk_graph = memory_builder.compile(checkpointer=saver)
            if not disk_graph.get_state(sqlite_cfg).values.get("messages", []):
                disk_graph.invoke(
                    {"messages": [HumanMessage(content="需落盘保存的教学记录")]},
                    config=sqlite_cfg,
                )
                print("首次写入。")
            else:
                print("读取到已有记录，没有重复追加。")
        with SqliteSaver.from_conn_string(str(db_path)) as saver:
            restored_graph = memory_builder.compile(checkpointer=saver)
            restored = restored_graph.get_state(sqlite_cfg).values["messages"]
            assert any(m.content == "需落盘保存的教学记录" for m in restored)
            print("关闭并重新连接数据库后恢复：通过；消息数 =", len(restored))
        print("数据库路径：", db_path)


SQLite 实验未开启；本格未写入数据库。


## 6.3 实战：把真实和风天气查询包装成 `@tool`

前半部分已经学过工具协议和执行循环；现在把命令行里跑通的天气查询接进同一条链路。

```text
用户：成都未来3天天气？
    → 模型生成 query_weather_forecast(city="成都", days=3)
    → 程序校验参数，调用已有 weather_tool.get_weather()
    → 本地读取配置、签发 JWT、发送 GET 请求、解析 JSON
    → ToolMessage 回填预报数据
    → 模型根据结果组织回答
```

`@tool` 生成工具名称、描述和参数 schema；`bind_tools()` 把这些信息交给模型。Python 工具的执行仍由第 4 节的执行器完成。

**本例的范围**：只支持成都、北京的本地示例坐标，返回每日预报；未启用城市搜索 GeoAPI，不能声称支持任意城市名。先把已有能力可靠接入，再按需要扩展城市查找。


In [57]:
import json
import sys
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import Literal

import jwt
import requests
from langchain_core.tools import tool, ToolException
from pydantic import BaseModel, ConfigDict, Field


def find_weather_dir(start: Path) -> Path:
    """从 Notebook 当前目录向上查找已有的天气工具，不依赖原 E 盘路径。"""
    for base in [start.resolve(), *start.resolve().parents]:
        for candidate in (base / "天气预报", base / "week24" / "天气预报", base):
            if (candidate / "weather_tool.py").is_file() and (candidate / "config.json").is_file():
                return candidate
    raise FileNotFoundError("未找到天气预报目录；请将 Notebook 与天气预报文件夹放在同一课程目录")


WEATHER_DIR = find_weather_dir(Path.cwd())
if str(WEATHER_DIR) not in sys.path:
    sys.path.insert(0, str(WEATHER_DIR))
import weather_tool as qweather_api

# 使用已经验证过的示例坐标。城市名 -> 坐标属于本地映射，不是在线城市搜索。
# 当前凭据只启用天气预报，未调用 GeoAPI；暂只支持以下两个城市的示例位置。
WEATHER_LOCATIONS = {"成都": (30.67, 104.07), "北京": (39.92, 116.41)}


class RealWeatherArgs(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    city: Literal["成都", "北京"] = Field(description="查询城市，目前支持成都或北京的示例位置")
    days: int = Field(default=3, ge=1, le=10, strict=True, description="未来预报天数，1～10，默认3天")


@tool(args_schema=RealWeatherArgs)
def query_weather_forecast(city: Literal["成都", "北京"], days: int = 3) -> str:
    """通过和风天气 API 查询成都或北京未来1～10天的真实每日天气预报。

    返回预报时段、白天/夜间天气和最高/最低温及单位。这是每日预报，不是当前实况。
    只支持成都或北京的示例坐标；缺少城市先追问，其他城市先说明目前不支持。
    """
    latitude, longitude = WEATHER_LOCATIONS[city]
    try:
        # config.json 与私钥由程序读取，不属于模型的输入 schema。
        local_config = qweather_api.load_config(WEATHER_DIR / "config.json")
        raw = qweather_api.get_weather(local_config, latitude, longitude, days)
    except requests.RequestException as exc:
        raise ToolException("天气服务连接失败或超时，请稍后重试；未获得本次预报。") from exc
    except OSError as exc:
        raise ToolException("本地天气配置或私钥不可读，请检查天气预报目录。") from exc
    except jwt.PyJWTError as exc:
        raise ToolException("天气服务 JWT 生成失败，请检查本地认证配置。") from exc
    except (ValueError, RuntimeError) as exc:
        raise ToolException(f"天气查询失败：{exc}") from exc

    # 精简返回值，保留时间、单位与来源，避免将整份庞大 JSON 发给模型。
    try:
        forecast_days = [
            {
                "forecastStartTime": day["forecastStartTime"],
                "forecastEndTime": day["forecastEndTime"],
                "daytime": day.get("daytime", {}).get("condition", {}).get("text"),
                "nighttime": day.get("nighttime", {}).get("condition", {}).get("text"),
                "temperatureMin": day["temperatureMin"],
                "temperatureMax": day["temperatureMax"],
            }
            for day in raw["days"]
        ]
    except (KeyError, TypeError) as exc:
        raise ToolException("天气服务返回的数据结构与每日预报接口不符。") from exc
    return json.dumps(
        {
            "ok": True,
            "source": "QWeather",
            "data_type": "daily_forecast",
            "city": city,
            "location": {"latitude": latitude, "longitude": longitude, "basis": "本地示例坐标"},
            "queried_at": datetime.now(timezone.utc).isoformat(),
            "days": forecast_days,
            "attributions": raw.get("metadata", {}).get("attributions", []),
        },
        ensure_ascii=False,
    )


print("工具名称：", query_weather_forecast.name)
print("工具说明：", query_weather_forecast.description)
print("模型可见的参数：")
print(json.dumps(query_weather_forecast.args_schema.model_json_schema(), ensure_ascii=False, indent=2))


工具名称： query_weather_forecast
工具说明： 通过和风天气 API 查询成都或北京未来1～10天的真实每日天气预报。

    返回预报时段、白天/夜间天气和最高/最低温及单位。这是每日预报，不是当前实况。
    只支持成都或北京的示例坐标；缺少城市先追问，其他城市先说明目前不支持。
模型可见的参数：
{
  "additionalProperties": false,
  "properties": {
    "city": {
      "description": "查询城市，目前支持成都或北京的示例位置",
      "enum": [
        "成都",
        "北京"
      ],
      "title": "City",
      "type": "string"
    },
    "days": {
      "default": 3,
      "description": "未来预报天数，1～10，默认3天",
      "maximum": 10,
      "minimum": 1,
      "title": "Days",
      "type": "integer"
    }
  },
  "required": [
    "city"
  ],
  "title": "RealWeatherArgs",
  "type": "object"
}


### 6.3.1 输入、返回值与凭据各自放在哪里？

| 内容 | 由谁提供 | 是否出现在工具输入 schema |
|---|---|---|
| `city`、`days` | 模型从用户问题中提取；本地再次校验 | 是 |
| 天气、温度、时间、单位 | 和风天气 API 返回 | 否 |
| API Host、开发者/项目/凭据 ID、私钥 | 程序读取本地文件 | 否 |
| JWT | 原天气程序按本地配置自动生成 | 否 |

工具返回 JSON 字符串，保留查询时间、预报起止时间、单位与来源。模型不能把每日预报称为“当前实测温度”。调用工具不会自动覆盖之前的 JSON/TXT 文件；本例的工具返回值用于消息回填。

本节展示完整的 `@tool` 定义；相同定义也保存在 `天气预报/weather_langchain_tool.py`，以后在其他脚本里可以导入复用。模型使用的城市映射和参数范围与本节一致。

下面只检查 schema 和非法参数，不访问天气或模型服务。


In [58]:
# 这一格不访问网络，验证模型看到的 schema 和本地参数校验。
from unittest.mock import patch
from pydantic import ValidationError
from langchain_core.utils.function_calling import convert_to_openai_tool

real_weather_schema = convert_to_openai_tool(query_weather_forecast)
parameters = real_weather_schema["function"]["parameters"]
assert set(parameters["properties"]) == {"city", "days"}
assert parameters["required"] == ["city"]
assert parameters["properties"]["city"]["enum"] == ["成都", "北京"]
assert parameters["properties"]["days"]["minimum"] == 1
assert parameters["properties"]["days"]["maximum"] == 10

with patch.object(qweather_api, "get_weather") as forbidden_request:
    for bad_args in ({}, {"city": "上海"}, {"city": "成都", "days": 0},
                     {"city": "成都", "days": 11}, {"city": "成都", "days": True},
                     {"city": "成都", "temperature": 99}):
        try:
            query_weather_forecast.invoke(bad_args)
        except ValidationError:
            pass
        else:
            raise AssertionError(f"错误参数未被拦截：{bad_args}")
    forbidden_request.assert_not_called()
print("真实天气工具 schema 与参数校验：通过；本格未发送天气请求。")


真实天气工具 schema 与参数校验：通过；本格未发送天气请求。


### 6.3.2 先直接调用工具，检查真实天气链路

把下一格 `RUN_LIVE_WEATHER` 改为 `True`，运行一次。这一步使用 `.invoke()` 执行真实的 `@tool`，只访问和风天气，**不经过语言模型**。成功后再做下一步。

开关默认关闭，避免“运行全部”时意外产生请求。前半部分的 `RUN_LIVE_DEMOS` 可以继续保持 `False`。


In [59]:
RUN_LIVE_WEATHER = False  # 改为 True 才会实际调用和风天气；不请求语言模型。
real_weather_result = None
if RUN_LIVE_WEATHER and os.environ.get("TOOL_NOTEBOOK_OFFLINE") != "1":
    real_weather_result = json.loads(query_weather_forecast.invoke({"city": "成都", "days": 3}))
    print(json.dumps(real_weather_result, ensure_ascii=False, indent=2))
else:
    print("真实天气查询未开启：将本格 RUN_LIVE_WEATHER 改为 True 后重新运行。")


真实天气查询未开启：将本格 RUN_LIVE_WEATHER 改为 True 后重新运行。


### 6.3.3 让真实模型选择工具，再根据结果回答

先运行第 0 节初始化以及第 3、4 节的工具与执行器定义，再运行本节。将下一格 `RUN_WEATHER_MODEL_DEMO` 改为 `True`，会单独读取原课程的 `week23/.env` 模型配置；前半部分仍可保持离线。

观察四件事：模型响应中的工具名与参数 → Python 工具的真实返回值 → 相同调用 ID 的 `ToolMessage` → 模型最终回答。`query_weather_forecast.invoke(...)` 只证明工具可以执行；这里完整跑通才是“模型请求工具，程序执行并回填，模型继续回答”。

第 4 节执行器只新增一个可选的 `system_prompt` 参数，默认值保持原模拟示例的行为；这里传入真实天气说明，避免沿用“天气均为模拟数据”的提示。

如需交给第 6 节框架管理，工具列表可以使用 `[query_weather_forecast]`，并把 `WEATHER_SYSTEM_PROMPT` 作为 `create_agent` 的 `system_prompt`。

参考：[LangChain 工具定义](https://docs.langchain.com/oss/python/langchain/tools)、[模型工具调用](https://docs.langchain.com/oss/python/langchain/models)、[和风天气每日预报](https://dev.qweather.com/docs/api/weather/weather-daily-forecast/)。


In [60]:
RUN_WEATHER_MODEL_DEMO = False  # 独立开关：开启后会产生模型用量，并可能调用天气 API。
weather_model_result = None
if RUN_WEATHER_MODEL_DEMO and os.environ.get("TOOL_NOTEBOOK_OFFLINE") != "1":
    # 复用第 0 节定义的 find_lab_dir；独立初始化，前半部分可保持默认离线。
    weather_lab_dir = str(find_lab_dir(Path.cwd()))
    if weather_lab_dir not in sys.path:
        sys.path.insert(0, weather_lab_dir)
    from lab_config import get_model

    weather_model = get_model(temperature=0)
    weather_bound_model = weather_model.bind_tools([query_weather_forecast])
    weather_today = datetime.now(timezone(timedelta(hours=8))).date().isoformat()
    WEATHER_SYSTEM_PROMPT = (
        f"北京时间今天是{weather_today}。你可以调用工具查询真实每日天气预报。"
        "查询天气时先调用工具，只依据返回的数据回答，并注明和风天气来源。"
        "目前只支持成都、北京的示例位置；缺少城市先追问，其他城市说明支持范围。"
        "每日预报不等于当前实况；保留温度单位，工具报错时如实说明，不编造天气。"
        "已有本次所需的预报结果后直接回答，不重复调用同一个查询。"
    )
    weather_model_result = run_tool_loop(
        "请调用天气工具查询成都未来3天的天气，按日期列出白天、夜间天气及最低和最高温。",
        max_steps=4,
        bound_model=weather_bound_model,
        tools_by_name={query_weather_forecast.name: query_weather_forecast},
        system_prompt=WEATHER_SYSTEM_PROMPT,
    )
    print("\n状态：", weather_model_result["status"])
    print("模型调用次数：", weather_model_result["model_calls"])
    print("最终回答：\n", weather_model_result["answer"])
    # 观察 messages 中实际有没有 tool_calls 和配对的 ToolMessage，不能只看最终文字。
else:
    print("模型调用真实天气的演示未开启：将本格 RUN_WEATHER_MODEL_DEMO 改为 True 后重新运行。")


模型调用真实天气的演示未开启：将本格 RUN_WEATHER_MODEL_DEMO 改为 True 后重新运行。


## 7. 收口：框架接管了什么，程序还要保证什么

| 环节 | 自定义协议 | 原生调用与 Agent |
|---|---|---|
| 告知工具能力 | prompt 中写说明 | 绑定工具 schema |
| 模型提出请求 | 正文 JSON，自行定义分支 | `tool_calls`，包含关联 id |
| 执行工具 | 自己分发 | 手写执行器或框架执行 |
| 回填结果 | 自定义约定 | `ToolMessage` 与请求 id 对应 |
| 推进任务 | 自己维护循环 | Agent 组织循环与状态 |
| 会话记忆 | 自己保存消息 | 历史存储或图检查点 |
| 正确性保障 | 校验、失败处理、次数上限与评估 | 同样需要配置与验证 |

### 原始课堂代码中的问题，以及本笔记的处理

1. `test4.py` 的退出判断比较了内置 `input`；应比较输入字符串。
2. `test4.py` 的历史获取函数返回了整个 store；应取指定会话。
3. `test3.py` 把天气和温度当成输入，且两份 schema 的 required 不一致；本笔记统一输入 city，结果由工具生成。
4. 多个原脚本返回注解写成 Weather，实际返回 str；本笔记按真实返回值标注。
5. `Field(example=...)` 是当前 Pydantic v2 中的弃用写法，使用 examples 或 json_schema_extra；不要写成“必然被忽略”。
6. `handle_parsing_errors` 应配置到 AgentExecutor，且仅覆盖解析错误。

上述修正在本 notebook 中完成，旁边的原始课堂脚本保留用于对照。

### 动手练习

1. 给天气输入加入 `unit: Literal["C", "F"] = "C"`，同时修改函数签名与返回值转换；比较默认值、枚举与 required。不要只改 schema。
2. 独立任务“北京和上海天气”与依赖任务“北京气温加 5”分别运行，画出实际消息顺序；不要预设模型一定同一轮发出多个调用。
3. 将 max_steps 设为 1，查看返回的 status 和 messages：最后可能已有工具结果，但还没有模型最终回答。
4. 让工具缺参、查询无记录、返回超时错误，确认错误关联到正确 id；再确认未知程序异常仍能被开发者看到。
5. 运行离线线程隔离检查；需要落盘时再开启 SQLite 实验，并亲自验证跨内核恢复。

### 自测问题

- bind_tools 做了什么？为什么它不会自动执行工具？
- 工具 schema 描述输入还是返回值？为什么不让模型填写待查询的温度？
- 一次模型响应带多个调用，是否表示 Python 已并发执行？
- 一个工具请求加一次最终回答，为什么一般至少需要两次模型请求？
- tool_call_id 能否防止重复调用？错误结果为什么也需要对应 id？
- 检查点恢复、短期历史、长期记忆与模型权重更新有什么区别？

### 检查边界与参考资料

- JSON/语法检查：证明文件结构与 Python 语法可读，不证明模型可用。
- 离线固定响应检查：证明给定轨迹下的执行、错误处理与状态分支，不证明模型能力。
- 真实 API 演示：需要自己开启，记录模型、版本、问题、完整轨迹与结果；一次成功不能代替系统评估。
- SQLite 实验：需要可选依赖并主动开启；没有执行时不声称已经验证落盘。

参考：[LangChain 模型与工具调用](https://docs.langchain.com/oss/python/langchain/models)、
[工具定义](https://docs.langchain.com/oss/python/langchain/tools)、
[Agent](https://docs.langchain.com/oss/python/langchain/agents)、
[短期记忆](https://docs.langchain.com/oss/python/langchain/short-term-memory)、
[LangGraph 检查点持久化](https://docs.langchain.com/oss/python/langgraph/persistence)、
[Pydantic JSON Schema](https://docs.pydantic.dev/latest/concepts/json_schema/)。
